In [2]:
import pandas as pd
import requests
from bs4 import BeautifulSoup


In [3]:

url = "https://merolagani.com/LatestMarket.aspx"

headers = {
    "User-Agent": "Mozilla/5.0"
}

response = requests.get(
    url,
    headers=headers,
    timeout=30
)

In [4]:
soup = BeautifulSoup(response.content, "html.parser")
print(soup.prettify())

<!DOCTYPE html>
<html lang="en" xmlns="http://www.w3.org/1999/xhtml">
 <head>
  <meta charset="utf-8"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <meta content="ca-pub-5212639010415313" name="google-adsense-account"/>
  <title>
   merolagani - Nepal Stock Exchange (NEPSE) Live Trading Data, Live Floorsheet, Live Indices, Top Gainers, Top Losers
  </title>
  <link as="style" href="/bundles/css/main200?v=XBmv3Nbj66DV6lzAxQCBgaIOrn903Diuut8lsSb3n9k1" rel="preload"/>
  <link as="style" href="/bundles/css/site205?v=L5V1uk5xoLic8qHQPSOAgTWwYTf5257cWEew3F97Y_c1" rel="preload"/>
  <link href="/bundles/css/main200?v=XBmv3Nbj66DV6lzAxQCBgaIOrn903Diuut8lsSb3n9k1" rel="stylesheet">
   <link href="/bundles/css/site205?v=L5V1uk5xoLic8qHQPSOAgTWwYTf5257cWEew3F97Y_c1" rel="stylesheet">
    <link href="/Content/Services.css" rel="stylesheet"/>
    <link href="/Content/owl.carousel.min.css" rel="stylesheet"/>
    <l

In [5]:
# Find the Live Trading table
table = soup.select_one(
    'div#ctl00_ContentPlaceHolder1_LiveTrading table.live-trading'
)

# Check whether table was found
if table:
    print("Table found successfully!")
else:
    print("Table not found!")

Table found successfully!


In [6]:
import pandas as pd

# Get column names
headers = []

for th in table.find_all("th"):
    headers.append(th.get_text(strip=True))

print(headers)

data = []

for row in table.find("tbody").find_all("tr"):

    cells = row.find_all("td")


    if len(cells) >= 7:

        symbol = cells[0].get_text(strip=True)
        ltp = cells[1].get_text(strip=True)
        change = cells[2].get_text(strip=True)
        open_price = cells[3].get_text(strip=True)
        high = cells[4].get_text(strip=True)
        low = cells[5].get_text(strip=True)
        qty = cells[6].get_text(strip=True)
        

        data.append([
            symbol,
            ltp,
            change,
            open_price,
            high,
            low,
            qty
        ])

['Symbol', 'LTP', '% Change', 'Open', 'High', 'Low', 'Qty.', 'PClose', 'Diff.', '', '']


In [7]:
# Create NEW DataFrame
df = pd.DataFrame(
    data,
    columns=[
        "Symbol",
        "LTP",
        "% Change",
        "Open",
        "High",
        "Low",
        "Qty."
    ]
    )

# Clean numeric columns
numeric_columns = [
    "LTP",
    "% Change",
    "Open",
    "High",
    "Low",
    "Qty."
]

for col in numeric_columns:

    df[col] = (
        df[col]
        .str.replace(",", "", regex=False)
    )

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

# Calculate previous closing price
df["PClose"] = (
    df["LTP"] /
    (1 + df["% Change"] / 100)
)

# Round to 2 decimal places
df["PClose"] = df["PClose"].round(2)

print(df)

print("\nNEW DATAFRAME:")
print(df)

print("\nShape:")
print(df.shape)

print("\nData Types:")
print(df.dtypes)

     Symbol     LTP  % Change    Open    High     Low    Qty.   PClose
0    ACLBSL   943.0      0.00   945.0   915.1   915.1     182   943.00
1      ADBL   304.8      0.26   305.8   302.0   302.0   70052   304.01
2       AHL   413.0     -0.84   415.1   407.0   415.0    1853   416.50
3      AHPC   265.9      0.72   271.0   264.3   265.0   77596   264.00
4     AKJCL   363.0     -0.22   369.8   361.1   364.0  229686   363.80
..      ...     ...       ...     ...     ...     ...     ...      ...
342    USLB  1121.5      0.58  1140.0  1121.0  1135.0     626  1115.03
343    VLBS   658.1     -0.59   665.0   650.3   650.3    2737   662.01
344   VLUCL   413.0     -0.46   416.0   410.5   416.0    4982   414.91
345    WNLB  1332.0      1.29  1339.0  1316.0  1316.0     120  1315.04
346    YMHL   623.0     -0.48   630.0   618.0   626.0   13280   626.00

[347 rows x 8 columns]

NEW DATAFRAME:
     Symbol     LTP  % Change    Open    High     Low    Qty.   PClose
0    ACLBSL   943.0      0.00   945.0

In [8]:
# Sector-wise script lists
Bank = ["NABIL", "NIMB", "SCB", "HBL", "SBI", "EBL", "NICA", "MBL", "LSL", "KBL",
        "SBL", "SANIMA", "NMB", "PRVU", "GBIME", "CZBIL", "PCBL", "ADBL", "NBL"]

manufacturing = ["BNL", "NLO", "BNT", "UNL", "HDL", "SHIVM", "GCIL", "SONA",
                 "SARBTM", "OMPL", "SAGAR", "SAIL", "SYPNL", "RSML", "PCIL", "SOPL", "ECL"]

hotel_and_tourism = ["SHL", "TRH", "OHL", "CGH", "KDL", "CITY", "BANDIPUR", "HFIN"]

other = ["NTC", "NRIC", "NRM", "MKCL", "NWCL", "HRL", "PURE", "TTL"]

hydropower = [
    "NHPC", "BPCL", "CHCL", "AHPC", "SHPC", "RIDI", "BARUN", "API",
    "NGPL", "KKHC", "DHPL", "AKPL", "SPDL", "UMHL", "CHL", "HPPL",
    "NHDL", "RADHI", "PMHPL", "KPCL", "AKJCL", "JOSHI", "UPPER", "GHL",
    "UPCL", "MHNL", "PPCL", "HURJA", "UNHPL", "RHPL", "SJCL", "HDHPC",
    "LEC", "SSHL", "MEN", "UMRH", "GLH", "SHEL", "RURU", "MKJC",
    "SAHAS", "TPC", "SPC", "NYADI", "MBJC", "BNHC", "GVL", "BHL",
    "RFPL", "DORDI", "BHDC", "HHL", "UHEWA", "SGHC", "MHL", "USHEC",
    "RHGCL", "SPHL", "PPL", "SIKLES", "EHPL", "PHCL", "BHPL", "SMHL",
    "SPL", "SMH", "MKHC", "AHL", "TAMOR", "MHCL", "SMJC", "MAKAR",
    "MKHL", "DOLTI", "BEDC", "MCHL", "IHL", "MEL", "RAWA", "USHL",
    "TSHL", "KBSH", "MEHL", "ULHC", "MANDU", "BGWT", "MSHL", "MMKJL",
    "TVCL", "VLUCL", "CKHL", "SANVI", "BHCL", "HIMSTAR", "MABEL", "DHEL",
    "BUNGAL", "SOHL", "BJHL", "SKHL", "RLEL", "SKHEL", "SIPD", "KHPL",
    "APHL", "YMHL", "TPKHL", "SNORL", "SGHL", "KAHL", "MEPDL"
]

trading = ["STC", "BBC"]

non_life_insurance = [
    "NICL", "RBCL", "HEI", "UAIL", "SPIL", "NIL", "PRIN",
    "SALICO", "IGI", "SICL", "NLG", "SGIC", "NMIC"
]

development_bank = [
    "NABBC", "EDBL", "LBBL", "MDB", "MLBL", "GBBL", "JBBL", "CORBL",
    "KSBBL", "SADBL", "SHINE", "MNBBL", "SINDU", "GRDBL", "SAPDBL", "SABBL"
]

finance = [
    "NFS", "GUFL", "BFC", "GFCL", "SIFC", "CFCL", "JFL",
    "GMFIL", "ICFC", "PROFL", "MPFL", "MFIL", "RLFL"
]

microfinance = [
    "NUBL", "CBBL", "DDBL", "SWBBL", "NMLBBL", "FMDBL", "SLBBL", "SKBBL",
    "GBLBS", "KMCDB", "MLBBL", "LLBS", "VLBS", "HLBSL", "MATRI", "JSLBB",
    "NMBMF", "GILB", "SWMF", "MERO", "NMFBS", "RSDC", "FOWAD", "SMATA",
    "MSLB", "SMB", "USLB", "WNLB", "NADEP", "ACLBSL", "SLBSL", "ALBSL",
    "GMFBS", "GLBSL", "SMFBS", "ILBS", "NICLBSL", "SMPDA", "MLBSL", "JBLB",
    "MLBS", "NESDO", "ULBSL", "CYCL", "AVYAN", "DLBS", "SHLB", "UNLB",
    "ANLB", "SWASTIK"
]

life_insurance = [
    "NLICL", "NLIC", "LICN", "ALICL", "HLI", "SJLIC", "PMLI",
    "SRLI", "ILI", "RNLI", "SNLI", "CLI", "GMLI", "CREST"
]

investment = [
    "CIT", "HIDCL", "NRN", "NIFRA", "CHDC", "ENL", "HATHY"
]
mutual_fund = [
    "SEF", "NBF2", "SIGS2", "NICBF", "NMB50", "SFMF", "LUK", "SLCF",
    "KEF", "SBCF", "PSF", "NIBSF2", "NICSF", "RMF1", "MMF1", "NBF3",
    "NICFC", "KDBY", "GIBF1", "NSIF2", "NIBLGF", "SAGF", "SFEF", "PRSF",
    "RMF2", "SIGS3", "C30MF", "LVF2", "H8020", "NICGF2", "KSY", "NIBLSTF",
    "MNMF1", "GSY", "NMBHF2", "MBLEF", "RSY", "GBIMESY2", "HLICF", "RBBF40",
    "CSY", "NSY", "SEF2", "SAEF2", "LSH12", "RSY2"
]

preference_share = [
    "NABILPNP",
    "KSBBLPNP",
    "SBLPNP",
    "NMBPNP",
    "SANIMAPNP",
    "MBLPNP"
]

# Create sector mapping
sector_mapping = {}

for script in Bank:
    sector_mapping[script] = "Banking"

for script in manufacturing:
    sector_mapping[script] = "Manufacturing"

for script in hotel_and_tourism:
    sector_mapping[script] = "Hotel and Tourism"

for script in other:
    sector_mapping[script] = "Other"

for script in hydropower:
    sector_mapping[script] = "Hydropower"

for script in trading:
    sector_mapping[script] = "Trading"

for script in non_life_insurance:
    sector_mapping[script] = "Non-Life Insurance"

for script in development_bank:
    sector_mapping[script] = "Development Bank"

for script in finance:
    sector_mapping[script] = "Finance"

for script in microfinance:
    sector_mapping[script] = "Microfinance"

for script in life_insurance:
    sector_mapping[script] = "Life Insurance"

for script in investment:
    sector_mapping[script] = "Investment"

for script in mutual_fund:
    sector_mapping[script] = "Mutual Fund"

for script in preference_share:
    sector_mapping[script] = "Preference Share"

# Create sector column
df["sector"] = df["Symbol"].map(sector_mapping)

In [9]:
from pathlib import Path
from datetime import date

# Go from src/scraper -> project root
project_root = Path.cwd().parents[1]

# data/raw
raw_folder = project_root / "data" / "raw"/"stock_data"

# Create folder if it doesn't exist
raw_folder.mkdir(parents=True, exist_ok=True)

# Filename
today_date = date.today().strftime("%Y-%m-%d")
file_path = raw_folder / f"{today_date}-stock_data.csv"

# Save
df.to_csv(file_path, index=False)

print(f"Saved successfully: {file_path}")

Saved successfully: e:\nepse-market-report\data\raw\stock_data\2026-08-13-stock_data.csv
